# Memorization extraction: Llama-3.1-8B

Runs LeakPro's `memorization_extraction` attack on Llama-3.1-8B, verifying generations against a
20 GB RedPajama sample indexed in Elasticsearch:

| | Setting |
|---|---|
| Config | `audit.yaml` (section 3) |
| Model | `meta-llama/Llama-3.1-8B` (`target_model.Llama31_8B`); bfloat16 on Gaudi and recent GPUs, float16 on older GPUs |
| Hardware | NVIDIA GPU (CUDA) or Intel Gaudi (HPU), about 24 GB of device memory or more |
| Reference corpus | 20 GB RedPajama-Data-1T sample, Elasticsearch index `redpajama_chunks` |
| Prefix pool | `data/prefixes_conditional.jsonl` (seed 42) |
| Phase 1 | baseline, temp-decay (T 10→1), conditional; 10,000 each, 256 tokens; top 5% by zlib and lowercase; 30-word windows, stride 5 |
| Phase 2 | anchors ≤ 50 tokens, prefixes 30-50, 150 generations each, T 3→1; novelty against Phase 1's exhaustive windows |
| Phase 3 *(optional)* | LoRA r=16 on attention + MLP, 5 epochs, batch 1 × 8 accumulation, lr 1e-4 |
| Phase 4 *(optional)* | Phase 1 strategies on the tuned model; 30-token overlap removal before filtering |
| Seed | 42 |

**Before running**
- **First time?** Follow section 0 below: Python environment, access to the gated Llama model,
  the reference corpus and prefix pool, and Elasticsearch. Each step is done once.
- Already have a prefix pool and an index? Set `target.data_path` in `audit.yaml` to the
  prefix pool and `ES_ENV` in section 1 to your Elasticsearch settings, and skip section 0.

Phases 1-2 are the base attack and always run. Phases 3-4 are an optional, more extensive attack
(fine-tuning with the weights): include them by uncommenting `finetune` in `audit.yaml`.

This is a long run (many hours for 30,000 generations; Phases 3-4 add fine-tuning and another
30,000). It is resumable:
re-running cell 4a and the phase cells continues from `leakpro_output_extraction/artifacts`.

## 0. Setup (first time only)

Run the shell commands in a terminal, from this folder (`examples/extraction/llm_memorization`),
in the order below. Steps 3 and 5 need step 2 (the model's tokenizer) and step 4 (a running
Elasticsearch).

### Step 1. Python environment

```bash
pip install -e "../../..[extraction]"   # LeakPro + torch, transformers, elasticsearch, peft
pip install zstandard                    # needed to download the Common Crawl part of the corpus
```

**NVIDIA GPU:** the install above brings a CUDA build of PyTorch on Linux. Check with
`python -c "import torch; print(torch.cuda.is_available())"`.

**Intel Gaudi:** use an environment that already has Habana's PyTorch (for example a
`leakpro_hpu` conda environment) and do not let pip replace `torch`.

Pick that environment as the notebook kernel. LeakPro uses Gaudi if present, else the GPU, else
the CPU; `LEAKPRO_DEVICE=cuda` (or `hpu`, `cpu`) chooses explicitly.

### Step 2. Access to `meta-llama/Llama-3.1-8B` (gated model)

1. Create a Hugging Face account, open <https://huggingface.co/meta-llama/Llama-3.1-8B>, fill in
   the form and accept the Llama 3.1 Community License. Meta approves the request, usually within
   hours; the status is shown at <https://huggingface.co/settings/gated-repos>.
2. Create an access token at <https://huggingface.co/settings/tokens>: type **Read** (or a
   fine-grained token with "Read access to contents of all public gated repos you can access").
3. Log in once on the machine; the token is stored in `~/.cache/huggingface/token`:
   ```bash
   hf auth login            # older huggingface_hub versions: huggingface-cli login
   ```
   Alternatively `export HF_TOKEN=...` before starting Jupyter. Never paste the token into this
   notebook.
4. The weights (about 16 GB) download automatically on the first run, into
   `~/.cache/huggingface` (set `HF_HOME` to use another disk). To download them up front:
   ```bash
   hf download meta-llama/Llama-3.1-8B --exclude "original/*"
   ```

Section 2 below checks that the access works.

### Step 3. Reference corpus and prefix pool

The corpus is a 20 GB sample of RedPajama-Data-1T (Wikipedia, StackExchange, C4, Common Crawl).
It is public: no account is needed. The download streams it from the dataset's own URLs and takes
several hours depending on bandwidth; it needs about 20 GB free.

```bash
python prepare_data.py download --target-gb 20 --output data/aux_corpus.jsonl
python prepare_data.py prefixes --model meta-llama/Llama-3.1-8B \
    --corpus data/aux_corpus.jsonl --output data/prefixes_conditional.jsonl --seed 42
```

The prefix step tokenizes with Llama's tokenizer, so it needs the access from step 2.

### Step 4. Elasticsearch

Phase 1-2 verification searches the 20 GB corpus through an Elasticsearch 8.x index. Install it as
a system service from Elastic's package repository (Debian/Ubuntu; needs `sudo`):

```bash
wget -qO - https://artifacts.elastic.co/GPG-KEY-elasticsearch \
    | sudo gpg --dearmor -o /usr/share/keyrings/elasticsearch-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/elasticsearch-keyring.gpg] https://artifacts.elastic.co/packages/8.x/apt stable main" \
    | sudo tee /etc/apt/sources.list.d/elastic-8.x.list
sudo apt-get update && sudo apt-get install elasticsearch
sudo systemctl daemon-reload
sudo systemctl enable --now elasticsearch      # starts now and after every reboot
```

The package turns on security (password and TLS) by itself. Set a password for the `elastic` user
and copy the CA certificate to a place your user can read:

```bash
sudo /usr/share/elasticsearch/bin/elasticsearch-reset-password -u elastic   # prints a password
mkdir -p ~/.elasticsearch
sudo cp /etc/elasticsearch/certs/http_ca.crt ~/.elasticsearch/http_ca.crt
sudo chown "$USER" ~/.elasticsearch/http_ca.crt
curl --cacert ~/.elasticsearch/http_ca.crt -u elastic https://localhost:9200   # asks for the password; prints the version
```

Then put the connection settings in a file **outside the repository**, e.g. `~/es_env.sh`, and
`source` it in the terminal (the notebook can also read it, see section 1):

```bash
export ES_HOST=https://localhost:9200
export ES_USER=elastic
export ES_PASSWORD=...                       # from the reset-password command
export ES_CA_CERT=~/.elasticsearch/http_ca.crt
```

The index is larger than the corpus (chunks overlap, so every word is stored twice); leave
plenty of disk space for Elasticsearch's data directory, `/var/lib/elasticsearch` (set `path.data`
in `/etc/elasticsearch/elasticsearch.yml` to use another disk). `sudo systemctl status elasticsearch`
shows whether the service runs.

### Step 5. Build the index

```bash
source ~/es_env.sh
python prepare_data.py index --corpus data/aux_corpus.jsonl --index redpajama_chunks
```

This takes hours for 20 GB. Afterwards, continue with section 1.

## 1. Configuration file and Elasticsearch credentials

The audit configuration is the file `audit.yaml` in this folder (shown in section 3). The
prefix pool and the index name are read from it, so they are set in one place. `ES_ENV` is an
optional shell file with the `export ES_...=` lines of section 0, step 4.

In [ ]:
import os
from pathlib import Path

import yaml
from run_extraction import ES_KEYS, load_es_env

AUDIT_YAML = Path("audit.yaml")
config = yaml.safe_load(AUDIT_YAML.read_text())
PREFIXES = Path(config["target"]["data_path"])
ES_INDEX = config["audit"]["attack_list"][0]["attacker_verifier"]["index"]
ES_ENV = Path.home() / "es_env.sh"   # optional: ES_* variables already exported are used as they are

# Sets ES_HOST / ES_USER / ES_PASSWORD / ES_CA_CERT from ES_ENV without printing them;
# variables already set in the environment win.
load_es_env(ES_ENV)
print("config:", AUDIT_YAML)
print("prefix pool:", PREFIXES)
print("Elasticsearch index:", ES_INDEX)
print("Elasticsearch settings found:", [k for k in ES_KEYS if os.environ.get(k)])

## 2. Check the inputs

In [ ]:
# Each check prints the setup step to redo when it fails.
from huggingface_hub import auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

from leakpro.attacks.utils.extraction.elasticsearch_verifier import connect

MODEL = "meta-llama/Llama-3.1-8B"
try:
    auth_check(MODEL)
    print(f"Hugging Face: access to {MODEL} OK")
except GatedRepoError:
    print(f"Hugging Face: no access to {MODEL} yet. Request it and log in (section 0, step 2).")
except HfHubHTTPError as e:
    print(f"Hugging Face: cannot check access ({e.__class__.__name__}); log in with `hf auth login` "
          "(section 0, step 2). Offline is fine if the weights are already cached.")

if PREFIXES.is_file():
    with PREFIXES.open() as f:
        print("prefix pool:", sum(1 for _ in f), "prefixes")
else:
    print(f"prefix pool: {PREFIXES} not found. Build it (section 0, step 3) or set target.data_path in audit.yaml.")

missing = [k for k in ("ES_HOST", "ES_USER", "ES_PASSWORD") if not os.environ.get(k)]
if missing:
    print(f"Elasticsearch: {missing} not set. See section 0, step 4.")
else:
    es = connect(ca_certs=os.environ.get("ES_CA_CERT"))
    if es.indices.exists(index=ES_INDEX):
        print("Elasticsearch index", ES_INDEX, "->", es.count(index=ES_INDEX)["count"], "chunks")
    else:
        print(f"Elasticsearch: index {ES_INDEX} not found. Build it (section 0, step 5).")

## 3. Audit configuration

As in the other LeakPro examples, the configuration is a separate YAML file, `audit.yaml`.
To change a setting, edit the file and re-run the notebook from section 1. Settings not listed use
the attack's defaults (README.md). `windowing: {min_span: 10}` also checks generations shorter
than 30 words, with one window over the whole text.

**Phases 3-4 are optional.** The file runs Phases 1-2, the base attack. For the more extensive
attack, uncomment `finetune: {}` in `audit.yaml` and re-run from section 1.

In [ ]:
print(AUDIT_YAML.read_text())

## 4. Run the audit, one phase per cell

Every phase takes hours, so each has its own cell. Run them in order. Each logs a line at most
once a minute with batches done, speed and time left.

**Resumable.** Every finished batch is cached under `leakpro_output_extraction/artifacts`. After an
interruption or a kernel restart, run 4a again and then the phases in order from 4b. Finished
phases and batches are read back from the cache in seconds, and the interrupted phase continues
where it stopped.

Cells 4b-4c are the base attack. Cells 4d-4e are the optional Phases 3-4: they run only when
`finetune` is set in the config (section 3), and otherwise say so and do nothing.

From a terminal instead (no notebook needed): `python run_extraction.py --phase 1`, then
`--phase 2` (optional: `--finetune --phase 3`, then 4); README.md, "Step by step in a terminal", lists every command.

### 4a. Load the model and prepare the attack (a few minutes)

In [ ]:
from run_extraction import prepare_attack

attack = prepare_attack(AUDIT_YAML)   # loads the model, seeds and prepares the attack (a few minutes)


def show(metrics):
    """One phase's counts per strategy as a table."""
    import pandas as pd
    table = pd.DataFrame(metrics).T.convert_dtypes()
    if "extraction_rate" in table:
        table["extraction_rate"] = (100 * table["extraction_rate"].astype(float)).round(2)
    return table.dropna(axis=1, how="all")

### 4b. Phase 1: initial extraction → S0

Baseline, temperature-decay and conditional generation, filtering, and verification against
Elasticsearch. The longest phase (many hours on one GPU or Gaudi card).

In [ ]:
show(attack.run_phase1())

### 4c. Phase 2: expansion → S1-novel

Re-prompts with anchors cut from S0 (150 generations per anchor), so its run time grows with |S0|.

In [ ]:
show(attack.run_phase2())

### 4d. Phase 3 (optional): LoRA fine-tuning on S0 ∪ S1-novel

The adapter stays attached for Phase 4. If the kernel restarts before Phase 4, the trained
adapter is read back from the cache.

In [ ]:
if attack.configs.finetune is not None:
    print(attack.run_phase3())
else:
    print("Phases 3-4 are not configured (optional): uncomment `finetune` in audit.yaml.")

### 4e. Phase 4 (optional): re-extraction with the fine-tuned model → S_new

Phase 1's strategies on the fine-tuned model; generations overlapping the fine-tuning text are
dropped. About as long as Phase 1. The adapter is removed afterwards.

In [ ]:
if attack.configs.finetune is not None:
    display(show(attack.run_phase4()))
else:
    print("Phases 3-4 are not configured (optional): uncomment `finetune` in audit.yaml.")

### 4f. Collect and save the result

Works after any phase: it reports the phases run so far. Writes
`leakpro_output_extraction/results/<attack id>/` (`result.json`, `extracted.jsonl`).

In [ ]:
result = attack.finish()
result.save(attack_obj=attack, output_dir=attack.output_dir)
print(result.summary())

## 5. Results

Per phase and strategy: generated outputs, candidates after
filtering (selected), verified memorized outputs and extraction rate (verified / generated).

In [ ]:
import pandas as pd

rows = [{"phase": phase, "strategy": strategy, **counts}
        for phase, by_strategy in result.metrics.items() for strategy, counts in by_strategy.items()]
table = pd.DataFrame(rows).set_index(["phase", "strategy"])
table["extraction_rate"] = (100 * table["extraction_rate"]).round(2)
table.dropna(axis=1, how="all")

In [ ]:
s0 = result.metrics["initial"]["total"]["verified"]
s1_novel = result.metrics.get("expansion", {}).get("total", {}).get("novel")
s_new = result.metrics.get("finetuned", {}).get("total", {}).get("verified")
print(f"S0 (Phase 1 verified):        {s0}")
print(f"S1-novel (Phase 2):           {s1_novel}")
print(f"S_new (Phase 4 verified):     {s_new}")

In [ ]:
# A few extracted sequences: the first verified window of each
for record in result.extracted[:5]:
    match = record["attacker_matches"][0]
    print(f"[{record['phase']}/{record['strategy']}] doc {match['doc_id']}")
    print("   ", match["window"][:200], "\n")